# EZ-VC Kaggle Gradio UI

This notebook clones EZ-VC on Kaggle, creates a Python 3.12 `.venv` with `uv`, installs the runtime there, and starts the inference or training Gradio UI with public sharing enabled.

Before running: add a Kaggle secret named `HF_TOKEN` with access to `SPRINGLab/EZ-VC`. You can also paste the token into the UI field after launch.

In [ ]:
from pathlib import Path
import os

WORKDIR = Path('/kaggle/working')
REPO_DIR = WORKDIR / 'EZ-VC'
os.environ['EZVC_REPO_URL'] = 'https://github.com/RahulBhalley/EZ-VC.git'
os.environ['EZVC_REPO_DIR'] = str(REPO_DIR)

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = token
    print('HF token loaded from Kaggle secret HF_TOKEN')
except Exception as exc:
    print('HF_TOKEN Kaggle secret not loaded:', exc)
    print('You can still paste the token into the Gradio HF Token field after launch.')

print('Repo directory:', REPO_DIR)

## Clone and install

This cell is idempotent. Re-run it after restarting a Kaggle session.

In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working
if [[ ! -d "${EZVC_REPO_DIR}/.git" ]]; then
  git clone "${EZVC_REPO_URL}" "${EZVC_REPO_DIR}"
else
  git -C "${EZVC_REPO_DIR}" pull --ff-only
fi

cd "${EZVC_REPO_DIR}"
python3 -m pip install -U uv
python3 -m uv python install 3.12
python3 -m uv venv --python 3.12 .venv
.venv/bin/python - <<'PY'
import sys
assert sys.version_info[:2] == (3, 12), sys.version
print('Using Python', sys.version)
PY
.venv/bin/python -m pip install -U pip setuptools wheel
.venv/bin/python -m pip install -e .
.venv/bin/python -m pip install torchcodec
.venv/bin/python -m pip install --no-deps 'espnet @ git+https://github.com/wanchichen/espnet.git@ssl'
.venv/bin/python -m pip install \
  configargparse typeguard humanfriendly librosa==0.9.2 jamo h5py kaldiio \
  torch_complex nltk g2p_en espnet_tts_frontend opt-einsum editdistance \
  sentencepiece resampy inflect distance more_itertools jaconv


## Start inference UI

Run this cell to open the EZ-VC inference UI. Gradio will print a public `gradio.live` URL. Stop the cell when you are done.

In [ ]:
%%bash
set -euo pipefail

cd "${EZVC_REPO_DIR}"
export HF_TOKEN="${HF_TOKEN:-}"
export HUGGING_FACE_HUB_TOKEN="${HUGGING_FACE_HUB_TOKEN:-${HF_TOKEN:-}}"
scripts/start_inference_gradio.sh --host 0.0.0.0 --port 7861 --venv .venv --share


## Start training UI

Run this cell instead when you want the fine-tuning UI. Gradio will print a public `gradio.live` URL. Stop the cell when you are done.

In [ ]:
%%bash
set -euo pipefail

cd "${EZVC_REPO_DIR}"
export HF_TOKEN="${HF_TOKEN:-}"
export HUGGING_FACE_HUB_TOKEN="${HUGGING_FACE_HUB_TOKEN:-${HF_TOKEN:-}}"
scripts/start_training_gradio.sh --host 0.0.0.0 --port 7862 --venv .venv --share
